# Dataset check — dataset_audit
Check pass on the raw files shared so I know I'm pulling the right data + also logging null counts so to stay reproducible.

In [1]:
from pathlib import Path
import pandas as pd
from hashlib import sha256

In [2]:
RAW_FILES = {
    'bsr_visual_data.csv': 'data/raw/bsr_visual_data.csv',
    '17k_products_amazon_data.csv': 'data/raw/17k_products_amazon_data.csv',
}
EXPECTED_COLUMNS = {
    'bsr_visual_data.csv': ['asin','item_name','keyword','brand','bsr_best','image_list'],
    '17k_products_amazon_data.csv': [
        'asin','item_name','keyword','sd_title','sd_customer_sentiments',
        'sd_ratings_distribution','sd_number_bought_past_month','bsr_best'
    ],
}
reports_dir = Path('reports')
reports_dir.mkdir(exist_ok=True)

In [4]:
records = []
for csv_name, rel_path in RAW_FILES.items():
    csv_path = Path(rel_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"raw file missing: {csv_path}")

    sha = sha256(csv_path.read_bytes()).hexdigest()  # keeping hash for reproducibility records
    df = pd.read_csv(csv_path, low_memory=False)

    missing_cols = [col for col in EXPECTED_COLUMNS[csv_name] if col not in df.columns]
    assert not missing_cols, f"{csv_name} missing columns: {missing_cols}"

    records.append({
        "file": csv_name,
        "path": str(csv_path),
        "sha256": sha,
        "rows": len(df),
        "n_columns": len(df.columns),
        "null_counts": df[EXPECTED_COLUMNS[csv_name]].isna().sum().to_dict(),
    })
records

FileNotFoundError: raw file missing: data/raw/bsr_visual_data.csv

In [ ]:
lines = ["# Report — dataset_audit", "", "## Raw files"]
for rec in records:
    lines.append(f"### {rec['file']}")
    lines.append(f"- path: `{rec['path']}`")
    lines.append(f"- sha256: `{rec['sha256']}`")
    lines.append(f"- rows: {rec['rows']}")
    lines.append(f"- columns: {rec['n_columns']}")
    lines.append("- null counts (key columns):")
    for col, val in rec['null_counts'].items():
        lines.append(f"  - {col}: {int(val)}")
    lines.append("")
report_path = reports_dir / 'dataset_audit.md'
report_path.write_text('\n'.join(lines))
report_path